# Piloto Semana 4 — Segundo modelo: Random Forest (sin tuning)

Notebook modular, mismo patrón que `02_Piloto_Baseline_RegresionLogistica.ipynb`, sobre
la misma partición piloto (`dataset_split_piloto.csv`, generada por
`01_Adquisicion_EDA.ipynb`).

**Por qué existe este notebook:** no es para comparar en serio contra el baseline —
eso es Fase 4 (semana 8, Protocolo v2.1 §11 P07, con tuning de Optuna). Este notebook
corre Random Forest con **hiperparámetros por defecto, sin tuning**, únicamente para
confirmar que el pipeline de preprocesamiento + entrenamiento + métricas es **agnóstico
al algoritmo** — que no quedó acoplado al baseline por accidente. Es una corrida piloto
adicional, no oficial.

**Evidencia que produce:** `Pilotaje/<entorno>/evidencias/random_forest/rf_config_piloto.json`
y `metricas_rf_piloto.json`.


In [1]:
import csv
import json
import platform
import time
from pathlib import Path

import pandas as pd
import psutil
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

SEED = 42
TARGET_COL = "Target"
NOMBRE_MODELO = "random_forest"


## 0. Entorno y rutas (mismo detector que el resto del piloto)

In [2]:
_entorno_detectado = {
    "Linux": "laptop_arch_linux",
    "Windows": "escritorio_windows10",
}.get(platform.system(), "entorno_no_reconocido")

RUTA_EVIDENCIAS_PILOTO = Path(f"../../Pilotaje/{_entorno_detectado}/evidencias")
RUTA_MODELO = RUTA_EVIDENCIAS_PILOTO / NOMBRE_MODELO
RUTA_MODELO.mkdir(parents=True, exist_ok=True)
RUTA_BITACORA = Path(f"../../Pilotaje/{_entorno_detectado}/bitacora_ejecucion.csv")

_proceso = psutil.Process()


def _siguiente_contador():
    """Continúa la numeración P-01, P-02... donde la haya dejado el último notebook
    que escribió en esta bitácora (01, 02 o 03), para que los pasos de los distintos
    notebooks del piloto no se pisen con el mismo id."""
    if RUTA_BITACORA.exists() and RUTA_BITACORA.stat().st_size > 0:
        with open(RUTA_BITACORA, newline="", encoding="utf-8") as f:
            ids = [fila["id"] for fila in csv.DictReader(f)]
        numeros = [int(i.split("-")[1]) for i in ids if i.startswith("P-")]
        return max(numeros) if numeros else 0
    return 0


_contador_pasos = [_siguiente_contador()]


class registrar_paso:
    """Igual que en los otros dos notebooks del piloto: mide tiempo y memoria (RSS,
    vía psutil, cross-platform) de un bloque y lo agrega a la bitácora compartida del
    entorno."""

    def __init__(self, accion, entrada, esperado, evidencia=""):
        self.accion, self.entrada, self.esperado, self.evidencia = accion, entrada, esperado, evidencia

    def __enter__(self):
        self.t0 = time.perf_counter()
        self.mem_inicio_mb = _proceso.memory_info().rss / 1024**2
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        _contador_pasos[0] += 1
        duracion = round(time.perf_counter() - self.t0, 3)
        mem_fin_mb = _proceso.memory_info().rss / 1024**2
        delta_mem_mb = round(mem_fin_mb - self.mem_inicio_mb, 1)
        observado = "OK" if exc_type is None else f"ERROR: {exc_val}"
        incidencia = "" if exc_type is None else str(exc_val)
        fila = {
            "id": f"P-{_contador_pasos[0]:02d}",
            "accion": self.accion,
            "entrada": self.entrada,
            "esperado": self.esperado,
            "observado": f"{observado} | tiempo={duracion}s | rss_actual={round(mem_fin_mb, 1)}MB | delta_mem={delta_mem_mb}MB",
            "incidencia": incidencia,
            "evidencia": self.evidencia,
        }
        existe_con_contenido = RUTA_BITACORA.exists() and RUTA_BITACORA.stat().st_size > 0
        with open(RUTA_BITACORA, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(fila.keys()))
            if not existe_con_contenido:
                w.writeheader()
            w.writerow(fila)
        print(f"[{fila['id']}] {self.accion} -> {fila['observado']}")
        return False


print(f"Entorno detectado: {_entorno_detectado}")
print(f"Evidencia de este modelo: {RUTA_MODELO}")


Entorno detectado: laptop_arch_linux
Evidencia de este modelo: ../../Pilotaje/laptop_arch_linux/evidencias/random_forest


## 1. Cargar la partición piloto

Generada por `01_Adquisicion_EDA.ipynb` (sección 8.1-8.2) — **no** la partición oficial
de Fase 3. Misma partición exacta que usó el baseline (02), para que los resultados de
ambos notebooks sean comparables entre sí dentro del piloto.

In [3]:
df_split_piloto = pd.read_csv(RUTA_EVIDENCIAS_PILOTO / "dataset_split_piloto.csv")

cols_features_piloto = [c for c in df_split_piloto.columns if c not in {TARGET_COL, "class_binaria", "split"}]

df_train_piloto = df_split_piloto[df_split_piloto["split"] == "train"]
df_val_piloto = df_split_piloto[df_split_piloto["split"] == "val"]

X_train_piloto = df_train_piloto[cols_features_piloto]
y_train_piloto = df_train_piloto["class_binaria"]
X_val_piloto = df_val_piloto[cols_features_piloto]
y_val_piloto = df_val_piloto["class_binaria"]

print(f"train={len(X_train_piloto)}  val={len(X_val_piloto)}  variables={len(cols_features_piloto)}")


train=840  val=180  variables=36


## 2. Preprocesamiento (P05, igual que en 01 y 02): escalado + SMOTE solo en train

Random Forest no necesita escalado en rigor (es invariante a la escala de las
variables), pero se mantiene el mismo preprocesamiento que el baseline a propósito —
el objetivo de este notebook es probar que **el pipeline** es agnóstico al algoritmo,
no optimizar el preprocesamiento por modelo (eso, si corresponde, es decisión de
Fase 4).

In [4]:
with registrar_paso(
    accion="Reajustar StandardScaler + SMOTE sobre train (mismo pipeline que 02)",
    entrada=f"train piloto ({len(X_train_piloto)} filas)",
    esperado="mismo resultado que en 01/02 (determinista, mismo seed)",
    evidencia="(no genera archivo nuevo; pipeline_config_piloto.json ya existe, generado por 01)",
):
    escalador = StandardScaler()
    X_train_escalado = escalador.fit_transform(X_train_piloto)
    X_train_proc, y_train_proc = SMOTE(random_state=SEED).fit_resample(X_train_escalado, y_train_piloto)
    X_val_proc = escalador.transform(X_val_piloto)

print(f"Train antes de SMOTE: {len(X_train_piloto)} -> después: {len(X_train_proc)}")


[P-07] Reajustar StandardScaler + SMOTE sobre train (mismo pipeline que 02) -> OK | tiempo=0.036s | rss_actual=208.5MB | delta_mem=3.7MB
Train antes de SMOTE: 840 -> después: 1022


## 3. Entrenar Random Forest — SIN tuning (hiperparámetros por defecto) — y métricas sobre validación

In [5]:
with registrar_paso(
    accion="Entrenar Random Forest (hiperparametros por defecto, sin tuning) y predecir sobre validación",
    entrada=f"train procesado ({len(X_train_proc)} filas tras SMOTE), val ({len(X_val_proc)} filas)",
    esperado="el modelo entrena sin error con el mismo pipeline que el baseline (02), sin tuning de hiperparámetros",
    evidencia=f"{NOMBRE_MODELO}/rf_config_piloto.json, {NOMBRE_MODELO}/metricas_rf_piloto.json",
):
    modelo_rf_piloto = RandomForestClassifier(random_state=SEED)  # hiperparámetros por defecto, sin tuning (eso es Fase 4)
    modelo_rf_piloto.fit(X_train_proc, y_train_proc)

    y_val_proba = modelo_rf_piloto.predict_proba(X_val_proc)[:, 1]
    y_val_pred = modelo_rf_piloto.predict(X_val_proc)

    metricas_rf_piloto = {
        "auc_roc": round(float(roc_auc_score(y_val_piloto, y_val_proba)), 4),
        "f1": round(float(f1_score(y_val_piloto, y_val_pred)), 4),
        "precision": round(float(precision_score(y_val_piloto, y_val_pred)), 4),
        "recall": round(float(recall_score(y_val_piloto, y_val_pred)), 4),
        "exactitud": round(float(accuracy_score(y_val_piloto, y_val_pred)), 4),
        "n_val": int(len(y_val_piloto)),
    }

    rf_config_piloto = {
        "modelo": "RandomForestClassifier",
        "hiperparametros": "por defecto de scikit-learn (sin tuning) + random_state=42",
        "nota": "Sin tuning de Optuna a propósito -- eso corresponde a Fase 4 (Protocolo v2.1 S11 P07). Esta corrida solo confirma que el pipeline funciona con un segundo algoritmo.",
        "smote": True,
        "seed": SEED,
    }
    with open(RUTA_MODELO / "rf_config_piloto.json", "w", encoding="utf-8") as f:
        json.dump(rf_config_piloto, f, indent=2, ensure_ascii=False)
    with open(RUTA_MODELO / "metricas_rf_piloto.json", "w", encoding="utf-8") as f:
        json.dump(metricas_rf_piloto, f, indent=2, ensure_ascii=False)

print(metricas_rf_piloto)


[P-08] Entrenar Random Forest (hiperparametros por defecto, sin tuning) y predecir sobre validación -> OK | tiempo=0.165s | rss_actual=210.0MB | delta_mem=1.5MB
{'auc_roc': 0.9545, 'f1': 0.8429, 'precision': 0.8551, 'recall': 0.831, 'exactitud': 0.8778, 'n_val': 180}


## 4. Comparación informal con el baseline (solo para este piloto, no es la comparación de Fase 6)

Carga las métricas que ya guardó `02_Piloto_Baseline_RegresionLogistica.ipynb` para
mostrar ambas una al lado de la otra. **No reemplaza** la comparación estadística
formal (Shapiro-Wilk → Friedman/Nemenyi → Wilcoxon/McNemar, Protocolo v2.1 §11 P11), que
corresponde a Fase 6 sobre los 4 modelos completos y la partición oficial.

In [6]:
ruta_metricas_baseline = RUTA_EVIDENCIAS_PILOTO / "baseline_regresion_logistica" / "metricas_baseline_piloto.json"

if ruta_metricas_baseline.exists():
    with open(ruta_metricas_baseline, encoding="utf-8") as f:
        metricas_baseline_piloto = json.load(f)
    comparacion = pd.DataFrame([
        {"modelo": "Regresión Logística (baseline)", **metricas_baseline_piloto},
        {"modelo": "Random Forest (sin tuning)", **metricas_rf_piloto},
    ]).set_index("modelo")
    print(comparacion)
else:
    print("No se encontró metricas_baseline_piloto.json -- correr primero 02_Piloto_Baseline_RegresionLogistica.ipynb")


                                auc_roc      f1  precision  recall  exactitud  \
modelo                                                                          
Regresión Logística (baseline)   0.9633  0.8652     0.8714  0.8592     0.8944   
Random Forest (sin tuning)       0.9545  0.8429     0.8551  0.8310     0.8778   

                                n_val  
modelo                                 
Regresión Logística (baseline)    180  
Random Forest (sin tuning)        180  
